# LLM Observability, Tracing & Prompt Management

Companion notebook for the [LLM Observability lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/09-llm-observability-and-prompt-management).

**The idea in one sentence.** An LLM request is a **trace** of **spans** (retrieval →
planning → tool → generation), and observability means measuring each span's **cost**,
**latency**, and **time-to-first-token** so you can find where the money and milliseconds
actually go.

What this notebook covers:

- **Traces & spans:** a request's total cost/latency is the sum over its spans — attribute
  it per stage.
- **Time-to-first-token (TTFT):** two systems with the same total latency feel very
  different if one streams the first token sooner.
- **Cost aggregation:** roll up the bill by feature to see what to optimise.

We build a tracing model from scratch and **validate the cost/latency attribution**, then
cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. A request is a trace of spans

One user request fans out into steps. We record each as a span with tokens, latency, and cost, then roll them up.

In [ ]:
PRICE_IN = 2 / 1_000_000   # $/input token
PRICE_OUT = 6 / 1_000_000  # $/output token

def span(name, tok_in, tok_out, seconds):
    cost = tok_in*PRICE_IN + tok_out*PRICE_OUT
    return {'name': name, 'tok_in': tok_in, 'tok_out': tok_out, 'sec': seconds, 'cost': cost}

trace = [
    span('retrieval',  120,   0, 0.05),
    span('planning',   800, 150, 0.9),
    span('tool:search', 60,  40, 0.4),
    span('generation', 1500, 600, 1.8),
]
for s in trace:
    print(f"{s['name']:12s} in={s['tok_in']:5d} out={s['tok_out']:4d} {s['sec']:.2f}s  ${s['cost']:.5f}")
print('-'*48)
print('request cost: $%.5f' % sum(s['cost'] for s in trace))
print('end-to-end latency: %.2fs' % sum(s['sec'] for s in trace))

### Validate: a trace's cost and latency are the sum of its spans

The whole point of tracing is attribution: the request's total cost and latency equal the
sum over its spans, so you can see which stage dominates. We confirm the totals and
identify the most expensive span.

In [ ]:
total_cost = sum(s['cost'] for s in trace)
total_sec = sum(s['sec'] for s in trace)
print(f'trace total: ${total_cost:.5f}, {total_sec:.2f} s over {len(trace)} spans')
for s in sorted(trace, key=lambda s: -s['cost']):
    print(f"  {s['name']:12s} ${s['cost']:.5f}  {s['sec']:.2f}s")
priciest = max(trace, key=lambda s: s['cost'])['name']
print(f'\nmost expensive span: {priciest}')
assert abs(total_cost - sum(s['tok_in']*PRICE_IN + s['tok_out']*PRICE_OUT for s in trace)) < 1e-12
assert priciest == 'generation', 'generation (most output tokens) should dominate the cost'
print('\n✅ tracing attributes cost/latency per span — you optimise what you can measure')

## 2. Time-to-first-token matters as much as total time

For a streaming UI, the user perceives latency at the **first token**, not the end. Two systems with the same total latency feel very different if one starts streaming sooner.

In [ ]:
# Two systems, same 3.0s total, different TTFT
sys_a = {'ttft': 0.4, 'total': 3.0}
sys_b = {'ttft': 2.2, 'total': 3.0}
fig, ax = plt.subplots(figsize=(8, 2.6))
for i, (name, s) in enumerate([('A', sys_a), ('B', sys_b)]):
    ax.barh(i, s['total'], color='#2e3347')
    ax.barh(i, s['ttft'], color=ROSE)
    ax.text(s['ttft']+0.05, i, f"TTFT {s['ttft']}s", va='center', fontsize=9, color='#e2e8f0')
ax.set_yticks([0,1]); ax.set_yticklabels(['system A', 'system B'])
ax.set_xlabel('seconds'); ax.set_title('Same total latency, very different perceived speed (TTFT in rose)')
plt.tight_layout(); plt.show()

### Validate: TTFT differs even at equal total latency

Two systems can have the *same* total latency but wildly different **time-to-first-token** —
and TTFT is what makes a stream feel responsive. We confirm the two systems match on total
but differ on TTFT (the metric a total-latency number hides).

In [ ]:
print(f"System A: TTFT {sys_a['ttft']}s, total {sys_a['total']}s")
print(f"System B: TTFT {sys_b['ttft']}s, total {sys_b['total']}s")
assert sys_a['total'] == sys_b['total'], 'same total latency'
assert sys_a['ttft'] < sys_b['ttft'], 'A streams the first token much sooner'
print(f"\nA feels {sys_b['ttft']/sys_a['ttft']:.1f}x more responsive despite identical total time")
print('✅ TTFT, not just total latency, determines perceived responsiveness')

## 3. Aggregate the bill by feature

Observability isn't just per-request — you aggregate cost across many requests to see *which feature* drives the bill.

In [ ]:
g = np.random.default_rng(1)
features = ['summarise', 'chat', 'rag_search']
rows = []
for _ in range(300):
    f = g.choice(features, p=[0.3, 0.5, 0.2])
    tin = {'summarise':3000,'chat':800,'rag_search':1500}[f] + g.integers(0,200)
    tout = {'summarise':400,'chat':300,'rag_search':500}[f] + g.integers(0,100)
    rows.append((f, tin*PRICE_IN + tout*PRICE_OUT))
cost_by_feature = {f: sum(c for ff,c in rows if ff==f) for f in features}
for f, c in sorted(cost_by_feature.items(), key=lambda x:-x[1]):
    print(f'{f:12s} ${c:.4f}')

fig, ax = plt.subplots(figsize=(7,3.8))
fs = list(cost_by_feature); cs = [cost_by_feature[f] for f in fs]
ax.bar(fs, cs, color=YELLOW); ax.set_ylabel('total cost ($)'); ax.set_title('Cost aggregated by feature')
ax.grid(True, alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **total latency hides TTFT** | a responsive stream needs low TTFT, not just low total (verified) |
| **no per-span attribution** | you can't optimise cost/latency you don't measure per stage |
| **cost blind spots** | a low-traffic token-heavy feature can dominate the bill (demo) |
| **prompt drift** | untracked prompt changes silently move cost/quality — version prompts |
| **sampling traces** | log enough traces to catch tail behaviour, not just averages |

Demo: rolling the bill up by feature reveals where spend concentrates.

In [ ]:
# Cost aggregation finds where the money goes: rolling the per-request bill up BY FEATURE
# reveals that a low-traffic-but-token-heavy feature can dominate the bill. We confirm the
# per-feature totals sum to the overall bill.
total_bill = sum(c for _, c in rows)
print('cost by feature:')
for f, c in sorted(cost_by_feature.items(), key=lambda x: -x[1]):
    print(f'  {f:12s} ${c:.4f}  ({100*c/total_bill:.0f}% of bill)')
assert abs(sum(cost_by_feature.values()) - total_bill) < 1e-9, 'per-feature costs sum to the total'
print('\nAttributing spend by feature shows what to optimise -> often a token-heavy minority feature.')

## ✏️ Your turn — request cost

Implement `request_cost(tok_in, tok_out, price_in, price_out)` returning the dollar cost of one call.

In [ ]:
def request_cost(tok_in, tok_out, price_in, price_out):
    """TODO(you): return tok_in*price_in + tok_out*price_out."""
    # TODO
    return ...


In [ ]:
c = request_cost(3000, 1000, 2/1_000_000, 6/1_000_000)
print('cost: $%.5f (expected $0.01200)' % c)
assert abs(c - 0.012) < 1e-9
assert request_cost(0, 0, 1, 1) == 0
print('✅ per-request cost model checks out.')

<details>
<summary>Solution</summary>

```python
def request_cost(tok_in, tok_out, price_in, price_out):
    return tok_in * price_in + tok_out * price_out
```

Attach this to every span, plus latency and quality, and log the *full resolved context* so failures are reproducible. Version prompts so you know which one served each trace.
</details>

## Recap

- A **trace** of nested **spans** localises failure and rolls up cost/latency in a multi-step app.
- Track **time-to-first-token**, not just total latency, for streaming UIs.
- Aggregate cost **by feature/user** to find what drives the bill.
- Manage prompts as **versioned, tested** artifacts tied to their traces.